In [ ]:
import os
import numpy as np
import cv2
import csv
from tqdm import tqdm

# Input and output folders
input_folder = r'D:\Tugas Akhir\Subject TA\S34\S34_FR_Frames'
output_folder = r'D:\Tugas Akhir\Subject TA\S34\S34_FR_MASK'

# CSV file containing mask coordinates
csv_file = r'C:\Users\hp\Downloads\S34FRCompressednotTrimmed.csv'

# Read the pixel coordinates from the CSV file
with open(csv_file, 'r') as file:
    csv_reader = csv.reader(file)
    header = next(csv_reader)  # Read the header row
    rows = list(csv_reader)

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Process frames and generate masks
for frame_number in tqdm(range(1, len(rows)+1), desc="Processing Frames", unit="frame"):
    # Load the image
    image_path = os.path.join(input_folder, f'Frame_{frame_number-1}.jpg')
    image = cv2.imread(image_path)

    if image is None:
        print(f"Unable to read {image_path}. Skipping...")
        continue

    # Get the image dimensions
    height, width = image.shape[:2]

    # Create a blank mask with the same dimensions as the image
    mask = np.zeros((height, width), dtype=np.uint8)

    # Read the corresponding row from the CSV file
    row = rows[frame_number-1]
    for i in range(0, len(row), 2):
        try:
            x, y = map(float, (row[i], row[i+1]))
            x, y = int(round(x)), int(round(y))

            # Check if the coordinates are within the image bounds
            if 0 <= x < width and 0 <= y < height:
                cv2.circle(mask, (x, y), 5, 255, -1)

        except ValueError:
            # Skip non-numeric values or handle them differently
            continue

    # Save the mask as a PNG file in the output folder
    output_path = os.path.join(output_folder, f'Frame_{frame_number-1}_mask.png')
    cv2.imwrite(output_path, mask)